In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split, StratifiedKFold,GroupShuffleSplit
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

import scripts.model_inference

## set up pyspark session

In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/16 12:31:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/16 12:31:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## set up config

In [ ]:
snapshot_date_str = "2024-01-01"
model_name = "credit_model_2024_09_01.pkl"


In [4]:
config = {}
config["snapshot_date_str"] = snapshot_date_str
config["snapshot_date"] = datetime.strptime(config["snapshot_date_str"], "%Y-%m-%d")
config["model_name"] = model_name
config["model_bank_directory"] = "model_bank/"
config["model_artefact_filepath"] = config["model_bank_directory"] + config["model_name"]

pprint.pprint(config)

{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 1, 1, 0, 0),
 'snapshot_date_str': '2024-01-01'}


## load model artefact from model bank

In [5]:
# Load the model from the pickle file
with open(config["model_artefact_filepath"], 'rb') as file:
    model_artefact = pickle.load(file)

print("Model loaded successfully! " + config["model_artefact_filepath"])

Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


## load feature store

In [6]:
# connect to feature store
folder_path = "datamart/gold/feature_store/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
features_store_sdf = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",features_store_sdf.count())

features_store_sdf.show()

row_count: 11974
+-----------+--------------------+---------------+------+--------+-------------------------+-------------+-------------+-----------------+---------------+-------------+-----------+--------------------+-------------------+----------------------+--------------------+--------------------+----------+----------------+------------------------+---------------------+-------------------+-----------------------+--------------------+---------------+-------------------------+---+-------------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+------+---------------------------+-------------------+--------------------+--------------------------+-----------------------+--------------------+-----------------------+-------------------+----------------------------+--------+----------------+----------------+-----------------------+
|Customer_ID|             loan_id|loan_start_date|tenure|loan_amt|Scheduled

## preprocess data for modeling

In [7]:
# --- load features for loans originated on the snapshot date (no label yet) ---
features_sdf = features_store_sdf.filter(col("snapshot_date") == config["snapshot_date"])
features_pdf = features_sdf.toPandas()
if features_pdf.empty:
    raise ValueError(f"No feature rows for snapshot {config['snapshot_date_str']}")

# --- preprocess with the transformer fitted in training ---
transformer_stdscaler = model_artefact["preprocessing_transformers"]["stdscaler"]
feature_cols = list(transformer_stdscaler.feature_names_in_)      # exact training columns, in training order

X_inference = transformer_stdscaler.transform(features_pdf[feature_cols])
print("X_inference", X_inference.shape)

# --- predict ---
model = model_artefact["model"]
y_inference = model.predict_proba(X_inference)[:, 1]

y_inference_pdf = features_pdf[["loan_id", "Customer_ID", "snapshot_date"]].copy()
y_inference_pdf["model_name"] = config["model_name"]
y_inference_pdf["model_predictions"] = y_inference

X_inference (485, 78)


In [8]:
y_inference_pdf

,loan_id,Customer_ID,snapshot_date,model_name,model_predictions
0,CUS_0x102d_2024_01_01,CUS_0x102d,2024-01-01,credit_model_2024_09_01.pkl,0.146800
1,CUS_0x1051_2024_01_01,CUS_0x1051,2024-01-01,credit_model_2024_09_01.pkl,0.141955
2,CUS_0x1269_2024_01_01,CUS_0x1269,2024-01-01,credit_model_2024_09_01.pkl,0.092763
3,CUS_0x1290_2024_01_01,CUS_0x1290,2024-01-01,credit_model_2024_09_01.pkl,0.113459
4,CUS_0x12d1_2024_01_01,CUS_0x12d1,2024-01-01,credit_model_2024_09_01.pkl,0.123603
...,...,...,...,...,...
480,CUS_0xc68d_2024_01_01,CUS_0xc68d,2024-01-01,credit_model_2024_09_01.pkl,0.116404
481,CUS_0xc6d8_2024_01_01,CUS_0xc6d8,2024-01-01,credit_model_2024_09_01.pkl,0.082588
482,CUS_0xe8d_2024_01_01,CUS_0xe8d,2024-01-01,credit_model_2024_09_01.pkl,0.751851
483,CUS_0xf9e_2024_01_01,CUS_0xf9e,2024-01-01,credit_model_2024_09_01.pkl,0.151008


## save model inference to datamart gold table

In [9]:
gold_directory = f"datamart/gold/model_predictions/{config["model_name"][:-4]}/"
print(gold_directory)

if not os.path.exists(gold_directory):
    os.makedirs(gold_directory)

# save gold table - IRL connect to database to write
partition_name = config["model_name"][:-4] + "_predictions_" + snapshot_date_str.replace('-','_') + '.parquet'
filepath = gold_directory + partition_name
spark.createDataFrame(y_inference_pdf).write.mode("overwrite").parquet(filepath)
# df.toPandas().to_parquet(filepath,
#           compression='gzip')
print('saved to:', filepath)

datamart/gold/model_predictions/credit_model_2024_09_01/


saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_01_01.parquet


## backfill

In [10]:
# set up config
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"

In [11]:
# generate list of dates to process
def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)


In [13]:
for snapshot_date in dates_str_lst:
    print(snapshot_date)
    scripts.model_inference.main(snapshot_date, model_name)

2023-01-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 1, 1, 0, 0),
 'snapshot_date_str': '2023-01-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


X_inference (530, 78)
datamart/gold/model_predictions/credit_model_2024_09_01/
saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_01_01.parquet


---completed job---


2023-02-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 2, 1, 0, 0),
 'snapshot_date_str': '2023-02-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


X_inference (501, 78)
datamart/gold/model_predictions/credit_model_2024_09_01/


saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2023_02_01.parquet


---completed job---


2023-03-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2023, 3, 1, 0, 0),
 'snapshot_date_str': '2023-03-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


X_inference (506, 78)
datamart/gold/model_predictions/credit_model_2024_09_01/


...

saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_10_01.parquet


---completed job---


2024-11-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 11, 1, 0, 0),
 'snapshot_date_str': '2024-11-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


X_inference (488, 78)
datamart/gold/model_predictions/credit_model_2024_09_01/


saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_11_01.parquet


---completed job---


2024-12-01


---starting job---


{'model_artefact_filepath': 'model_bank/credit_model_2024_09_01.pkl',
 'model_bank_directory': 'model_bank/',
 'model_name': 'credit_model_2024_09_01.pkl',
 'snapshot_date': datetime.datetime(2024, 12, 1, 0, 0),
 'snapshot_date_str': '2024-12-01'}
Model loaded successfully! model_bank/credit_model_2024_09_01.pkl


X_inference (515, 78)
datamart/gold/model_predictions/credit_model_2024_09_01/


saved to: datamart/gold/model_predictions/credit_model_2024_09_01/credit_model_2024_09_01_predictions_2024_12_01.parquet


---completed job---




## Check datamart

In [14]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

In [15]:
folder_path = "datamart/gold/model_predictions/credit_model_2024_09_01/"
files_list = [folder_path+os.path.basename(f) for f in glob.glob(os.path.join(folder_path, '*'))]
df = spark.read.option("header", "true").parquet(*files_list)
print("row_count:",df.count())

df.show()

row_count: 11974
+--------------------+-----------+-------------+--------------------+--------------------+
|             loan_id|Customer_ID|snapshot_date|          model_name|   model_predictions|
+--------------------+-----------+-------------+--------------------+--------------------+
|CUS_0xb9cd_2024_0...| CUS_0xb9cd|   2024-02-01|credit_model_2024...| 0.04990736395120621|
|CUS_0xba9f_2024_0...| CUS_0xba9f|   2024-02-01|credit_model_2024...|  0.5564172863960266|
|CUS_0xbabc_2024_0...| CUS_0xbabc|   2024-02-01|credit_model_2024...|   0.543403148651123|
|CUS_0xbb1b_2024_0...| CUS_0xbb1b|   2024-02-01|credit_model_2024...| 0.06664140522480011|
|CUS_0xbb26_2024_0...| CUS_0xbb26|   2024-02-01|credit_model_2024...|   0.169528067111969|
|CUS_0xbb33_2024_0...| CUS_0xbb33|   2024-02-01|credit_model_2024...|0.046635664999485016|
|CUS_0xbb69_2024_0...| CUS_0xbb69|   2024-02-01|credit_model_2024...| 0.04780182987451553|
|CUS_0xbd06_2024_0...| CUS_0xbd06|   2024-02-01|credit_model_2024...|  0.